 EcoCore — IBM LinuxONE Jupyter MVP (Phase 2)
 
 Run top to bottom with Kernel → Restart Kernel and Run All Cells.
 
 This notebook detects its own execution environment, reads aggregate local/container metrics (if available),
 trains a demonstrator risk classifier using synthetic data, and compares hypothetical workload decisions. No real 
 workload is moved or delayed. Model metrics do not constitute production validation.
 IBM container caveat: If JupyterLab is hosted inside a container, psutil may show host-wide, container-limited, 
 or mixed views depending on kernel/cgroup configuration. Do not assume the readings represent the entire physical IBM LinuxONE server.

In [2]:
import sys
import platform
import psutil

print("Python:", sys.version)
print("OS:", platform.platform())
print("Architecture:", platform. machine())
print("CPU cores:", psutil.cpu_count(logical=True))
print("Memory GB:", round(psutil.virtual_memory().total / (1024**3), 2))

Python: 3.12.15 (main, Oct  1 2026, 00:00:00) [GCC 14.3.1 20251022 (Red Hat 14.3.1-4)]
OS: Linux-6.8.0-147-generic-s390x-with-glibc2.39
Architecture: s390x
CPU cores: 3
Memory GB: 7.82


In [3]:
print("CPU:", psutil.cpu_percent(interval=1), "%")
print("Memory:", psutil.virtual_memory().percent, "%")
print("Processes:", len(psutil.pids()))

try:
    print("Load average:", psutil.getloadavg())
except AttributeError:
    print("Load average unavailable")

CPU: 1.3 %
Memory: 43.2 %
Processes: 6
Load average: (0.16015625, 0.033203125, 0.0107421875)


In [4]:
import psutil
from datetime import datetime

def collect_metrics():
    """
    Collect current host telemetry.
    Returns one clean dictionary for the rest of EcoCore.
    """
    
    cpu_pct = psutil.cpu_percent(interval=1)
    memory_pct = psutil.virtual_memory().percent

    load_1, load_5, load_15 = psutil.getloadavg()
    logical_cores = psutil.cpu_count(logical=True) or 1

    process_count = len(psutil.pids())

    load_per_core = load_1 / logical_cores

    return {
        "timestamp": datetime.now().isoformat(),
        "cpu_pct": round(cpu_pct, 2),
        "memory_pct": round(memory_pct, 2),
        "load_per_core": round(load_per_core, 3),
        "process_count": process_count,
        "logical_cores": logical_cores
    }

In [6]:
metrics = collect_metrics()
metrics

{'timestamp': '2026-10-17T09:13:27.306574',
 'cpu_pct': 1.7,
 'memory_pct': 43.2,
 'load_per_core': 0.041,
 'process_count': 6,
 'logical_cores': 3}

In [7]:
print("CPU:", metrics["cpu_pct"], "%")
print("Memory:", metrics["memory_pct"], "%")
print("Load/core:", metrics["load_per_core"])
print("Processes:", metrics["process_count"])
print("Logical cores:", metrics["logical_cores"])

CPU: 1.7 %
Memory: 43.2 %
Load/core: 0.041
Processes: 6
Logical cores: 3


In [8]:
import pandas as pd
import time

samples = []

print("Collecting telemetry for 60 seconds...")

start_time = time.time()

while time.time() - start_time < 60:
    metrics = collect_metrics()
    samples.append(metrics)

    print(
        f"CPU={metrics['cpu_pct']}% | "
        f"Memory={metrics['memory_pct']}% | "
        f"Load/Core={metrics['load_per_core']}"
    )

    time.sleep(5)

print(f"\nCollected {len(samples)} samples.")

CPU=1.0% | Memory=43.6% | Load/Core=0.027
CPU=0.3% | Memory=43.6% | Load/Core=0.024
CPU=1.3% | Memory=43.5% | Load/Core=0.022
CPU=0.7% | Memory=43.5% | Load/Core=0.019
CPU=0.3% | Memory=43.5% | Load/Core=0.017
CPU=1.3% | Memory=43.5% | Load/Core=0.042
CPU=0.3% | Memory=43.6% | Load/Core=0.039
CPU=0.3% | Memory=43.5% | Load/Core=0.036
CPU=0.3% | Memory=43.5% | Load/Core=0.03
CPU=0.7% | Memory=43.5% | Load/Core=0.055

Collected 10 samples.


In [9]:
telemetry_df = pd.DataFrame(samples)

telemetry_df.to_csv(
    "ecocore_telemetry.csv",
    index=False
)

telemetry_df

,timestamp,cpu_pct,memory_pct,load_per_core,process_count,logical_cores
0,2026-10-17T09:13:54.188840,1.0,43.6,0.027,6,3
1,2026-10-17T09:14:00.190034,0.3,43.6,0.024,6,3
2,2026-10-17T09:14:06.191166,1.3,43.5,0.022,6,3
3,2026-10-17T09:14:12.192372,0.7,43.5,0.019,6,3
4,2026-10-17T09:14:18.193640,0.3,43.5,0.017,6,3
5,2026-10-17T09:14:24.194724,1.3,43.5,0.042,6,3
6,2026-10-17T09:14:30.195955,0.3,43.6,0.039,6,3
7,2026-10-17T09:14:36.197132,0.3,43.5,0.036,6,3
8,2026-10-17T09:14:42.198165,0.3,43.5,0.030,6,3
9,2026-10-17T09:14:48.199326,0.7,43.5,0.055,6,3


In [10]:
telemetry_df["cpu_trend"] = telemetry_df["cpu_pct"].diff()
telemetry_df["memory_trend"] = telemetry_df["memory_pct"].diff()

telemetry_df

,timestamp,cpu_pct,memory_pct,load_per_core,process_count,logical_cores,cpu_trend,memory_trend
0,2026-10-17T09:13:54.188840,1.0,43.6,0.027,6,3,NaN,NaN
1,2026-10-17T09:14:00.190034,0.3,43.6,0.024,6,3,-0.7,0.0
2,2026-10-17T09:14:06.191166,1.3,43.5,0.022,6,3,1.0,-0.1
3,2026-10-17T09:14:12.192372,0.7,43.5,0.019,6,3,-0.6,0.0
4,2026-10-17T09:14:18.193640,0.3,43.5,0.017,6,3,-0.4,0.0
5,2026-10-17T09:14:24.194724,1.3,43.5,0.042,6,3,1.0,0.0
6,2026-10-17T09:14:30.195955,0.3,43.6,0.039,6,3,-1.0,0.1
7,2026-10-17T09:14:36.197132,0.3,43.5,0.036,6,3,0.0,-0.1
8,2026-10-17T09:14:42.198165,0.3,43.5,0.030,6,3,0.0,0.0
9,2026-10-17T09:14:48.199326,0.7,43.5,0.055,6,3,0.4,0.0


In [11]:
telemetry_df["cpu_trend"] = telemetry_df["cpu_pct"].diff()
telemetry_df["memory_trend"] = telemetry_df["memory_pct"].diff()

telemetry_df["cpu_avg"] = telemetry_df["cpu_pct"].rolling(3).mean()
telemetry_df["memory_avg"] = telemetry_df["memory_pct"].rolling(3).mean()

telemetry_df.tail()


,timestamp,cpu_pct,memory_pct,load_per_core,process_count,logical_cores,cpu_trend,memory_trend,cpu_avg,memory_avg
5,2026-10-17T09:14:24.194724,1.3,43.5,0.042,6,3,1.0,0.0,0.766667,43.500000
6,2026-10-17T09:14:30.195955,0.3,43.6,0.039,6,3,-1.0,0.1,0.633333,43.533333
7,2026-10-17T09:14:36.197132,0.3,43.5,0.036,6,3,0.0,-0.1,0.633333,43.533333
8,2026-10-17T09:14:42.198165,0.3,43.5,0.030,6,3,0.0,0.0,0.300000,43.533333
9,2026-10-17T09:14:48.199326,0.7,43.5,0.055,6,3,0.4,0.0,0.433333,43.500000


In [12]:
telemetry_df.to_csv(
    "ecocore_telemetry.csv",
    index=False
)

print("Telemetry file updated successfully.")

Telemetry file updated successfully.


In [13]:
print(telemetry_df.columns.tolist())

['timestamp', 'cpu_pct', 'memory_pct', 'load_per_core', 'process_count', 'logical_cores', 'cpu_trend', 'memory_trend', 'cpu_avg', 'memory_avg']


In [14]:
from telemetry import collect_metrics

metrics = collect_metrics()

print(metrics)

{'timestamp': '2026-10-17T09:20:25.258179', 'cpu_pct': 1.3, 'memory_pct': 43.6, 'load_per_core': 0.071, 'process_count': 6, 'logical_cores': 3}


In [15]:
for i in range(3):
    print(collect_metrics())

{'timestamp': '2026-10-17T09:20:59.346921', 'cpu_pct': 0.7, 'memory_pct': 43.5, 'load_per_core': 0.039, 'process_count': 6, 'logical_cores': 3}
{'timestamp': '2026-10-17T09:21:00.348035', 'cpu_pct': 0.0, 'memory_pct': 43.5, 'load_per_core': 0.039, 'process_count': 6, 'logical_cores': 3}
{'timestamp': '2026-10-17T09:21:01.348828', 'cpu_pct': 1.3, 'memory_pct': 43.6, 'load_per_core': 0.039, 'process_count': 6, 'logical_cores': 3}


In [16]:

from telemetry import collect_metrics

m1 = collect_metrics()
print(m1)

{'timestamp': '2026-10-17T09:25:47.062177', 'cpu_pct': 1.0, 'memory_pct': 43.7, 'load_per_core': 0.018, 'process_count': 6, 'logical_cores': 3}


In [17]:
m2 = collect_metrics()
print(m2)

{'timestamp': '2026-10-17T09:25:59.333812', 'cpu_pct': 0.3, 'memory_pct': 43.7, 'load_per_core': 0.014, 'process_count': 6, 'logical_cores': 3}
